# Sparse → Full Reconstruction with OmniField on CIFAR-10

**Protocol**

1. For each image, deterministically sample a **fixed 40% pool** of its 1024 pixels (seeded by image index, identical across train / test).
2. **Training:** at every iteration, randomly split the pool into two disjoint halves — **20 % input tokens** and **20 % supervision targets**. The model encodes the input pixels with the OmniField backbone and is queried at the target coordinates.
3. **Test:** use the *fixed* first / second half split. Two MSE metrics:
   - `MSE_held-out` — error on the held-out 20 % (the same metric the model trained on).
   - `MSE_full` — error on **all 1024 pixels** (full neural-field reconstruction from only 20 % input).

This stresses (a) generalization to unseen target coordinates within the fixed pool and (b) the continuous-field interpolation property of the OmniField decoder.


In [ ]:
import math, ssl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- experiment config ----
IMAGE_SIZE  = 32
N_PIX       = IMAGE_SIZE * IMAGE_SIZE          # 1024
FRAC_POOL   = 0.40                             # per-instance fixed pool
K_POOL      = int(round(FRAC_POOL * N_PIX))    # 410
K_HALF      = K_POOL // 2                      # 205 input / 205 target

CHANNELS    = 3
FOURIER_DIM = 96
POS_DIM     = FOURIER_DIM * 2                  # 192
INPUT_DIM   = CHANNELS + POS_DIM               # 195
QUERY_DIM   = POS_DIM                          # 192
LOGITS_DIM  = CHANNELS                         # 3

LATENT_DIMS = (256, 384, 512)
NUM_LATENTS = (256, 256, 256)

BATCH_SIZE  = 64
EPOCHS      = 30
LR          = 2e-4
POOL_SEED   = 12345

print(f"Device: {DEVICE}")
print(f"K_POOL = {K_POOL} px (40%) | K_HALF = {K_HALF} input / {K_HALF} target")


In [ ]:
class SparseCIFAR10(Dataset):
    """CIFAR-10 with a per-instance fixed 40% pixel pool.

    train=True  : random 20/20 split inside the pool, re-sampled per __getitem__.
    train=False : deterministic split (first half = input, second half = target).
    """
    def __init__(self, base_dataset, image_size=IMAGE_SIZE, frac_pool=FRAC_POOL,
                 seed=POOL_SEED, train=True):
        self.base   = base_dataset
        self.N_pix  = image_size * image_size
        self.K_pool = int(round(frac_pool * self.N_pix))
        self.k_half = self.K_pool // 2
        self.train  = train

        rng = np.random.RandomState(seed)
        self.pool_idx = np.stack(
            [rng.permutation(self.N_pix)[:self.K_pool] for _ in range(len(base_dataset))],
            axis=0,
        ).astype(np.int64)

        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, image_size),
            torch.linspace(-1.0, 1.0, image_size),
            indexing='ij',
        )
        self.coords_all = torch.stack([ys, xs], dim=-1).view(self.N_pix, 2)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _   = self.base[idx]
        pix_all  = img.permute(1, 2, 0).reshape(-1, 3)
        pool     = self.pool_idx[idx]

        if self.train:
            perm    = np.random.permutation(self.K_pool)
            in_idx  = pool[perm[:self.k_half]]
            tgt_idx = pool[perm[self.k_half:]]
        else:
            in_idx  = pool[:self.k_half]
            tgt_idx = pool[self.k_half:]

        return {
            "in_pixels":   pix_all[in_idx],
            "in_coords":   self.coords_all[in_idx],
            "tgt_pixels":  pix_all[tgt_idx],
            "tgt_coords":  self.coords_all[tgt_idx],
            "full_pixels": pix_all,
            "full_coords": self.coords_all,
            "in_idx":      torch.from_numpy(in_idx),
        }


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
cifar_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_ds = SparseCIFAR10(cifar_train, train=True)
test_ds  = SparseCIFAR10(cifar_test,  train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Sanity check shapes
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  {k:12s} {tuple(v.shape)}  {v.dtype}")


In [ ]:
# ---- OmniField backbone (single modality): GFF + sinusoidal-init learnable latents + ICMR ----

class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


def get_sinusoidal_embeddings(n, d):
    assert d % 2 == 0
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn   = fn
        self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None:
            kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)


class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return x * F.gelu(gates)


class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult * 2),
            GEGLU(),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim  = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        attn = sim.softmax(dim=-1)
        out  = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))


class CascadedBlock(nn.Module):
    """Perceiver block with sinusoidal-init learnable latents + ICMR residual from previous stage."""
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None):
        super().__init__()
        self.latents = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads,
                                                 dim_head=cross_dim_head), context_dim=input_dim)
        self.self_attn  = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff = PreNorm(dim, FeedForward(dim))
    def forward(self, context, residual=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        x = self.cross_attn(x, context=context) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x


class OmniFieldCIFAR(nn.Module):
    """Single-modality OmniField:
        3-stage cascaded ICMR encoder → 4-block self-attn trunk → decoder cross-attn → RGB head.
    """
    def __init__(self, input_dim=INPUT_DIM, queries_dim=QUERY_DIM, logits_dim=LOGITS_DIM,
                 latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128, self_heads=8, self_dim_head=128,
                 num_trunk_layers=4):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev)
            )
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([
                PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                PreNorm(final, FeedForward(final)),
            ]) for _ in range(num_trunk_layers)
        ])
        self.decoder_cross_attn = PreNorm(queries_dim,
            Attention(queries_dim, final, heads=cross_heads, dim_head=cross_dim_head),
            context_dim=final)
        self.decoder_ff = PreNorm(queries_dim, FeedForward(queries_dim))
        self.to_logits  = nn.Linear(queries_dim, logits_dim)

    def forward(self, data, queries):
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        x = self.decoder_cross_attn(queries, context=residual) + queries
        x = self.decoder_ff(x) + x
        return self.to_logits(x)


In [ ]:
fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0).to(DEVICE)
model           = OmniFieldCIFAR().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in fourier_encoder.parameters())
print(f"Parameters: {n_params/1e6:.2f}M")

optimizer = AdamW(list(model.parameters()) + list(fourier_encoder.parameters()), lr=LR)
loss_fn   = nn.MSELoss()
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))


def encode_input(in_pixels, in_coords):
    pos = fourier_encoder(in_coords)                  # (B, S_in, POS_DIM)
    return torch.cat([in_pixels, pos], dim=-1)        # (B, S_in, INPUT_DIM)


def encode_query(coords):
    return fourier_encoder(coords)                    # (B, S_q, POS_DIM)


In [ ]:
@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval(); fourier_encoder.eval()
    held_sum = full_sum = 0.0
    held_n   = full_n   = 0
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        in_p   = batch['in_pixels'].to(DEVICE)
        in_c   = batch['in_coords'].to(DEVICE)
        tgt_p  = batch['tgt_pixels'].to(DEVICE)
        tgt_c  = batch['tgt_coords'].to(DEVICE)
        full_p = batch['full_pixels'].to(DEVICE)
        full_c = batch['full_coords'].to(DEVICE)

        data      = encode_input(in_p, in_c)
        pred_held = model(data, encode_query(tgt_c))
        pred_full = model(data, encode_query(full_c))

        held_sum += F.mse_loss(pred_held, tgt_p,  reduction='sum').item(); held_n += tgt_p.numel()
        full_sum += F.mse_loss(pred_full, full_p, reduction='sum').item(); full_n += full_p.numel()
    return held_sum / max(held_n, 1), full_sum / max(full_n, 1)


def visualize_eval(epoch, n_show=6):
    model.eval(); fourier_encoder.eval()
    batch  = next(iter(test_loader))
    in_p   = batch['in_pixels'][:n_show].to(DEVICE)
    in_c   = batch['in_coords'][:n_show].to(DEVICE)
    in_idx = batch['in_idx'][:n_show]
    full_p = batch['full_pixels'][:n_show].to(DEVICE)
    full_c = batch['full_coords'][:n_show].to(DEVICE)

    with torch.no_grad():
        recon = model(encode_input(in_p, in_c), encode_query(full_c))
    recon  = recon.clamp(-1, 1).view(n_show, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2)
    orig   = full_p.view(n_show, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2)

    # Sparse-input panel: black where unobserved
    sparse = torch.full_like(orig, -1.0)
    for b in range(n_show):
        idx = in_idx[b]
        ys, xs = idx // IMAGE_SIZE, idx % IMAGE_SIZE
        sparse[b, :, ys, xs] = orig[b, :, ys, xs]

    grid = torch.cat([orig, sparse, recon], dim=0)
    grid = (grid * 0.5 + 0.5).clamp(0, 1)
    grid = torchvision.utils.make_grid(grid.cpu(), nrow=n_show, padding=2)
    plt.figure(figsize=(2.2 * n_show, 7))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(f"Epoch {epoch}: orig (top) | 20% input, black=unobserved (mid) | full reconstruction (bot)")
    plt.axis('off'); plt.tight_layout(); plt.show()


# ---- Training loop ----
history = {'train': [], 'val_held': [], 'val_full': []}

for epoch in range(1, EPOCHS + 1):
    model.train(); fourier_encoder.train()
    epoch_loss, steps = 0.0, 0
    for batch in train_loader:
        in_p  = batch['in_pixels'].to(DEVICE)
        in_c  = batch['in_coords'].to(DEVICE)
        tgt_p = batch['tgt_pixels'].to(DEVICE)
        tgt_c = batch['tgt_coords'].to(DEVICE)

        data    = encode_input(in_p, in_c)
        queries = encode_query(tgt_c)
        pred    = model(data, queries)
        loss    = loss_fn(pred, tgt_p)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item(); steps += 1

    avg_train          = epoch_loss / max(steps, 1)
    val_held, val_full = evaluate(model, test_loader, max_batches=50)
    history['train'].append(avg_train)
    history['val_held'].append(val_held)
    history['val_full'].append(val_full)
    print(f"[Epoch {epoch:02d}/{EPOCHS}] train={avg_train:.4f}  "
          f"val_held20%={val_held:.4f}  val_full100%={val_full:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        visualize_eval(epoch)

print("\nDone.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ep = range(1, len(history['train']) + 1)
ax.plot(ep, history['train'],    label='train MSE (20% target)')
ax.plot(ep, history['val_held'], label='val MSE (held-out 20%)')
ax.plot(ep, history['val_full'], label='val MSE (full 100%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.set_yscale('log')
ax.set_title('Sparse → Full Reconstruction with OmniField (CIFAR-10)')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()
